# Global Paymaster IT Jobs — Germany Focus

**Goal:** Discover IT / Technology / Software career openings at *global* Employer-of-Record (EOR) and Paymaster vendors that employ people in Germany.

| Priority | Geography |
|----------|-----------|
| ★ Primary | Baden-Württemberg (Stuttgart, Karlsruhe, Mannheim, Heidelberg, Freiburg …) |
| ○ Secondary | All other German states + Germany-wide remote roles |

**Public APIs queried (no authentication required):**
- Greenhouse Job Board API (`boards-api.greenhouse.io`)
- Lever Postings API (`api.lever.co`)
- Workable Widget API (`apply.workable.com`)

**Output:** Timestamped `.xlsx` workbook with 9 colour-coded worksheets + inline statistics in this notebook.

---
**Run all cells top-to-bottom.** Prerequisites: `pip install requests pandas openpyxl`

In [31]:
# Uncomment the line below to install dependencies on first run:
# !pip install requests pandas openpyxl

import re
import time
import unicodedata
import warnings
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

import requests
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.utils.dataframe import dataframe_to_rows

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 90)
pd.set_option('display.max_rows', 200)

print('\u2713 Libraries loaded.')

✓ Libraries loaded.


## 1 · Configuration & Company Registry

In [32]:
# ── Output filename (timestamped) ────────────────────────────────────────────
OUTPUT_XLSX = f"global_paymaster_IT_jobs_Germany_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"

# ── Germany location keywords ─────────────────────────────────────────────────
GERMANY_ALIASES = [
    'germany', 'deutschland', 'remote - germany', 'remote (germany)',
    'remote, germany', ', germany', '/ germany', '- germany', '– germany',
]

# ── Baden-Württemberg keywords (ASCII + umlaut variants) ──────────────────────
BW_KEYWORDS = [
    'baden-wuerttemberg', 'baden wuerttemberg', 'badenwuerttemberg',
    'baden-württemberg', 'baden württemberg', 'bw', 'b-w',
    'stuttgart', 'karlsruhe', 'mannheim', 'heidelberg', 'freiburg',
    'ulm', 'heilbronn', 'reutlingen', 'tübingen', 'tuebingen',
    'pforzheim', 'konstanz', 'offenburg', 'esslingen',
    'böblingen', 'boeblingen', 'ludwigsburg', 'sinsheim',
    'ravensburg', 'friedrichshafen', 'aalen', 'sindelfingen',
    'waiblingen', 'schwäbisch gmünd', 'schwaebisch gmuend', 'nürtingen',
]

# ── IT / Technology role detection ───────────────────────────────────────────
TECH_INCLUDE_RE = re.compile(
    r'\b(software|engineer|engineering|developer|development|devops|sre|'
    r'platform|data|analytics|machine.?learning|ml|ai|'
    r'artificial.?intelligence|security|infosec|cybersecurity|cyber|'
    r'cloud|infrastructure|it|sysadmin|systems.?admin|network|'
    r'qa|quality.?assurance|test|automation|architect|'
    r'site.?reliability|backend|back.?end|frontend|front.?end|'
    r'fullstack|full.?stack|mobile|ios|android|flutter|'
    r'database|dba|etl|bi|business.?intelligence|'
    r'observability|kubernetes|k8s|docker|mlops|'
    r'erp|sap|crm|scrum.?master|agile.?delivery)\b',
    re.IGNORECASE
)
TECH_EXCLUDE_RE = re.compile(
    r'\b(sales|account.?executive|business.?development|'
    r'marketing|brand|communications|'
    r'hr|people.?operations|recruiting|recruiter|talent.?acquisition|'
    r'customer.?success|customer.?support|customer.?experience|'
    r'payroll.?specialist|payroll.?manager|payroll.?admin|'
    r'legal|compliance.?officer|general.?counsel|'
    r'finance|accounting|accounts.?payable|accounts.?receivable)\b',
    re.IGNORECASE
)
TECH_ALLOWLIST_RE = re.compile(
    r'\b(payroll.?engineer|software.?engineer|data.?engineer|'
    r'security.?engineer|platform.?engineer|cloud.?engineer|'
    r'network.?engineer|devops.?engineer|systems.?engineer|'
    r'ml.?engineer|ai.?engineer|site.?reliability.?engineer)\b',
    re.IGNORECASE
)

# ── Company Source Registry ───────────────────────────────────────────────────
# 'hq' = fallback location string applied to jobs that come back with no
#         location field (useful for companies whose ATS omits location data).
#         Leave '' to keep the job's own location as-is.
SOURCES: List[Dict[str, str]] = [
    # ── Greenhouse Job Board API — board tokens VERIFIED WORKING ─────────────
    {'company': 'G-P (Globalization Partners)', 'type': 'greenhouse',
     'id': 'globalizationpartners',
     'url': 'https://www.globalization-partners.com/careers/', 'hq': ''},

    # ── Lever Postings API — site slugs VERIFIED WORKING ─────────────────────
    {'company': 'RemoFirst',  'type': 'lever',    'id': 'remofirst',
     'url': 'https://jobs.lever.co/remofirst',  'hq': ''},

    # ── Workable Widget API — account slugs VERIFIED WORKING ─────────────────
    # WorkMotion is a German company (Munich HQ). Their Workable job listings
    # return blank location strings, so we fill in 'Germany (Remote)' as the
    # fallback so Germany-filter and IT-filter can classify them correctly.
    {'company': 'WorkMotion', 'type': 'workable', 'id': 'workmotion',
     'url': 'https://apply.workable.com/workmotion', 'hq': 'Germany (Remote)'},

    # ── Manual portals (Workday / proprietary ATS — no public scraping API) ──
    {'company': 'Deel',             'type': 'manual', 'id': '', 'hq': '',
     'url': 'https://www.deel.com/careers/'},
    {'company': 'Remote',           'type': 'manual', 'id': '', 'hq': '',
     'url': 'https://remote.com/careers'},
    {'company': 'Rippling',         'type': 'manual', 'id': '', 'hq': '',
     'url': 'https://www.rippling.com/careers'},
    {'company': 'Oyster HR',        'type': 'manual', 'id': '', 'hq': '',
     'url': 'https://www.oysterhr.com/careers'},
    {'company': 'Velocity Global',  'type': 'manual', 'id': '', 'hq': '',
     'url': 'https://velocityglobal.com/company/careers/'},
    {'company': 'Omnipresent',      'type': 'manual', 'id': '', 'hq': '',
     'url': 'https://omnipresent.com/careers/'},
    {'company': 'Atlas HXM',        'type': 'manual', 'id': '', 'hq': '',
     'url': 'https://www.atlashxm.com/careers'},
    {'company': 'Papaya Global',    'type': 'manual', 'id': '', 'hq': '',
     'url': 'https://www.papayaglobal.com/careers/'},
    {'company': 'Multiplier',       'type': 'manual', 'id': '', 'hq': '',
     'url': 'https://www.usemultiplier.com/careers'},
    {'company': 'Safeguard Global', 'type': 'manual', 'id': '', 'hq': '',
     'url': 'https://safeguardglobal.wd3.myworkdayjobs.com/External_Careers/'},
    {'company': 'CloudPay',         'type': 'manual', 'id': '', 'hq': '',
     'url': 'https://cloudpay.wd3.myworkdayjobs.com/CloudPay_External'},
    {'company': 'TMF Group',        'type': 'manual', 'id': '', 'hq': '',
     'url': 'https://www.tmf-group.com/en/careers/'},
    {'company': 'SD Worx',          'type': 'manual', 'id': '', 'hq': '',
     'url': 'https://careers.sdworx.com/jobs'},
    {'company': 'Lano',             'type': 'manual', 'id': '', 'hq': '',
     'url': 'https://www.lano.io/careers'},
    {'company': 'Airswift',         'type': 'manual', 'id': '', 'hq': '',
     'url': 'https://www.airswift.com/careers'},
    {'company': 'Horizons',         'type': 'manual', 'id': '', 'hq': '',
     'url': 'https://www.joinhorizons.com/careers'},
    {'company': 'ADP',              'type': 'manual', 'id': '', 'hq': '',
     'url': 'https://jobs.adp.com/'},
]

api_count    = sum(1 for s in SOURCES if s['type'] != 'manual')
manual_count = sum(1 for s in SOURCES if s['type'] == 'manual')
print(f'✓ Config ready: {len(SOURCES)} companies '
      f'({api_count} with verified public API, {manual_count} manual-only).')
print(f'✓ Output file : {OUTPUT_XLSX}')

✓ Config ready: 20 companies (3 with verified public API, 17 manual-only).
✓ Output file : global_paymaster_IT_jobs_Germany_20260513_1226.xlsx


## 2 · HTTP Utilities & Classification Helpers

In [33]:
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Corporate proxies perform TLS inspection and replace server certificates.
# verify=False bypasses the resulting SSL error for these public job-board APIs.
SESSION = requests.Session()
SESSION.headers.update({
    'User-Agent': 'Mozilla/5.0 (compatible; PaymasterJobCollector/2.0)'
})
SESSION.verify = False


def http_get_json(url: str, params: Optional[Dict] = None, timeout: int = 30) -> Any:
    r = SESSION.get(url, params=params, timeout=timeout)
    r.raise_for_status()
    return r.json()


def normalize(s: Optional[str]) -> str:
    if not s:
        return ''
    s = unicodedata.normalize('NFKC', str(s))
    for fancy in ('‑', '–', '—'):
        s = s.replace(fancy, '-')
    return s.strip()


def is_germany(location_blob: str) -> bool:
    t = location_blob.lower()
    return (
        any(alias in t for alias in GERMANY_ALIASES)
        or t.startswith('germany')
        or ' germany' in t
        or t == 'de'
    )


def is_bw(location_blob: str) -> bool:
    t = location_blob.lower()
    return any(k in t for k in BW_KEYWORDS)


def region_bucket(location_blob: str) -> str:
    if not is_germany(location_blob):
        return 'Non-Germany'
    if is_bw(location_blob):
        return 'Baden-Württemberg'
    t = location_blob.lower()
    if 'remote' in t:
        return 'Germany — Remote / Unspecified'
    return 'Germany — Other State'


def is_it_role(title: str, dept: str = '', team: str = '', snippet: str = '') -> bool:
    blob = normalize(' '.join([title, dept, team, snippet]))
    if TECH_INCLUDE_RE.search(blob) is None:
        return False
    if TECH_ALLOWLIST_RE.search(blob):
        return True
    if TECH_EXCLUDE_RE.search(blob):
        return False
    return True


print('✓ Helpers defined  (SSL verification disabled for corporate proxy).')

✓ Helpers defined  (SSL verification disabled for corporate proxy).


## 3 · Job Board Fetchers

In [34]:
_EMPTY_ROW: Dict[str, str] = {
    'company': '', 'source_system': '', 'title': '',
    'department': '', 'team': '', 'location': '', 'office': '',
    'remote_type': '', 'country_code': '',
    'apply_url': '', 'job_url': '', 'updated_at': '',
    'description_snippet': '',
}


def fetch_greenhouse(board_id: str, company: str) -> List[Dict]:
    url  = f'https://boards-api.greenhouse.io/v1/boards/{board_id}/jobs'
    data = http_get_json(url, params={'content': 'true'})
    jobs = data.get('jobs', []) if isinstance(data, dict) else []
    rows = []
    for j in jobs:
        depts  = j.get('departments') or []
        dept   = ', '.join(d.get('name', '') for d in depts  if isinstance(d, dict) and d.get('name'))[:400]
        offices = j.get('offices') or []
        office = ', '.join(o.get('name', '') for o in offices if isinstance(o, dict) and o.get('name'))[:400]
        rows.append({**_EMPTY_ROW,
            'company':             company,
            'source_system':       'greenhouse',
            'title':               normalize(j.get('title')),
            'department':          dept,
            'office':              office,
            'location':            normalize((j.get('location') or {}).get('name')),
            'apply_url':           normalize(j.get('absolute_url')),
            'job_url':             normalize(j.get('absolute_url')),
            'updated_at':          normalize(j.get('updated_at')),
            'description_snippet': normalize(j.get('content', ''))[:1200],
        })
    return rows


def fetch_lever(site: str, company: str) -> List[Dict]:
    base  = f'https://api.lever.co/v0/postings/{site}'
    rows  = []
    skip, limit = 0, 100
    while True:
        data = http_get_json(base, params={'mode': 'json', 'skip': skip, 'limit': limit})
        if not isinstance(data, list) or not data:
            break
        for j in data:
            cats    = j.get('categories') or {}
            job_url = normalize(j.get('hostedUrl'))
            rows.append({**_EMPTY_ROW,
                'company':             company,
                'source_system':       'lever',
                'title':               normalize(j.get('text')),
                'department':          normalize(cats.get('department')),
                'team':                normalize(cats.get('team')),
                'location':            normalize(cats.get('location')),
                'remote_type':         normalize(j.get('workplaceType')),
                'country_code':        normalize(j.get('country')),
                'apply_url':           normalize(j.get('applyUrl')) or job_url,
                'job_url':             job_url,
                'description_snippet': normalize(j.get('descriptionPlain', ''))[:1200],
            })
        if len(data) < limit:
            break
        skip += limit
        time.sleep(0.25)
    return rows


def fetch_workable(account: str, company: str) -> List[Dict]:
    url  = f'https://apply.workable.com/api/v1/widget/accounts/{account}'
    data = http_get_json(url)
    jobs = data.get('jobs', []) if isinstance(data, dict) else []
    rows = []
    for j in jobs:
        loc     = normalize(j.get('location'))
        job_url = normalize(j.get('url'))
        desc    = normalize(j.get('short_description') or j.get('description') or '')
        rows.append({**_EMPTY_ROW,
            'company':             company,
            'source_system':       'workable',
            'title':               normalize(j.get('title')),
            'department':          normalize(j.get('department')),
            'location':            loc,
            'remote_type':         'remote' if 'remote' in loc.lower() else '',
            'apply_url':           normalize(j.get('application_url')) or job_url,
            'job_url':             job_url,
            'description_snippet': desc[:1200],
        })
    return rows


print('\u2713 Fetchers defined (Greenhouse / Lever / Workable).')

✓ Fetchers defined (Greenhouse / Lever / Workable).


## 4 · Data Collection Pipeline

In [35]:
def collect_all_jobs() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Fetch from all API sources; log results; collect manual portals."""
    all_rows:    List[Dict] = []
    manual_rows: List[Dict] = []
    log_rows:    List[Dict] = []

    for src in SOURCES:
        company = src['company']
        stype   = src['type']
        sid     = src['id']
        portal  = src['url']
        hq      = src.get('hq', '')   # fallback location for blank-location jobs

        if stype == 'manual':
            manual_rows.append({
                'Company':    company,
                'Portal URL': portal,
                'ATS Type':   'Manual / Workday / Proprietary',
                'Note':       'No verified public API. Search portal manually for IT roles in Germany.',
            })
            continue

        try:
            if stype == 'greenhouse':
                rows = fetch_greenhouse(sid, company)
            elif stype == 'lever':
                rows = fetch_lever(sid, company)
            elif stype == 'workable':
                rows = fetch_workable(sid, company)
            else:
                rows = []
                manual_rows.append({'Company': company, 'Portal URL': portal,
                                    'ATS Type': stype, 'Note': 'Unknown source type.'})

            # Apply HQ location to jobs that came back with no location data
            if hq:
                for row in rows:
                    if not row.get('location') and not row.get('office') and not row.get('country_code'):
                        row['location'] = hq

            all_rows.extend(rows)
            log_rows.append({'Company': company, 'API Type': stype,
                             'Jobs Fetched': len(rows), 'Status': 'OK'})
            print(f'  ✓ {company:<42} [{stype:<10}]  {len(rows):>4} jobs')

        except Exception as exc:
            err = repr(exc)[:180]
            log_rows.append({'Company': company, 'API Type': stype,
                             'Jobs Fetched': 0, 'Status': f'ERROR: {err}'})
            manual_rows.append({
                'Company':    company,
                'Portal URL': portal,
                'ATS Type':   stype,
                'Note':       f'API fetch failed — check portal manually. Error: {err}',
            })
            print(f'  ✗ {company:<42} [{stype:<10}]  ERROR: {repr(exc)[:70]}')

    if all_rows:
        df_jobs = (
            pd.DataFrame(all_rows)
              .drop_duplicates(subset=['company', 'title', 'location'], keep='first')
              .reset_index(drop=True)
        )
    else:
        df_jobs = pd.DataFrame(columns=list(_EMPTY_ROW.keys()))

    df_manual = pd.DataFrame(manual_rows)
    df_log    = pd.DataFrame(log_rows)
    return df_jobs, df_manual, df_log


def enrich(df: pd.DataFrame) -> pd.DataFrame:
    """Add Germany / BW / IT classification columns."""
    df = df.copy()
    df['location_blob'] = (
        df['location'].fillna('') + ' ' +
        df['office'].fillna('')   + ' ' +
        df['country_code'].fillna('')
    ).str.strip()

    df['Is Germany']            = df['location_blob'].apply(is_germany)
    df['Is Baden-Württemberg']  = df['location_blob'].apply(is_bw)
    df['Region']                = df['location_blob'].apply(region_bucket)
    df['Is IT / Tech Role']     = df.apply(lambda r: is_it_role(
        str(r.get('title', '')),
        str(r.get('department', '')),
        str(r.get('team', '')),
        str(r.get('description_snippet', '')),
    ), axis=1)
    df['Collected (UTC)'] = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
    return df


print('✓ Pipeline functions defined.')

✓ Pipeline functions defined.


In [36]:
print('=' * 65)
print('  FETCHING JOBS FROM PUBLIC APIs')
print('=' * 65)

df_raw, df_manual, df_log = collect_all_jobs()
df_all = enrich(df_raw)

germany_mask = df_all['Is Germany']
bw_mask      = df_all['Is Baden-Württemberg']
it_mask      = df_all['Is IT / Tech Role']

print()
print('─' * 65)
print(f'  Total jobs collected (after dedup)  : {len(df_all):>5,}')
print(f'  IT / Tech roles — all locations     : {it_mask.sum():>5,}')
print(f'  Germany jobs (all roles)            : {germany_mask.sum():>5,}')
print(f'  Germany IT / Tech roles             : {(germany_mask & it_mask).sum():>5,}')
print(f'  Baden-Württemberg IT roles  ★       : {(bw_mask & it_mask).sum():>5,}')
print(f'  Manual portals listed               : {len(df_manual):>5,}')
print('─' * 65)

if not df_log.empty:
    print('\n── API Fetch Log ──')
    print(df_log.to_string(index=False))

if not df_all.empty:
    print(f'\n── Sample Raw Data (first 10 rows) ──')
    cols = ['company', 'title', 'location', 'Is Germany', 'Is IT / Tech Role']
    print(df_all[[c for c in cols if c in df_all.columns]].head(10).to_string(index=False))
else:
    print('\n⚠  No jobs collected. All API calls failed or returned empty results.')
    print('   Check the Fetch_Log sheet in the Excel file for error details.')

  FETCHING JOBS FROM PUBLIC APIs
  ✓ G-P (Globalization Partners)               [greenhouse]    43 jobs
  ✓ RemoFirst                                  [lever     ]    13 jobs
  ✓ WorkMotion                                 [workable  ]    17 jobs

─────────────────────────────────────────────────────────────────
  Total jobs collected (after dedup)  :    61
  IT / Tech roles — all locations     :    21
  Germany jobs (all roles)            :     5
  Germany IT / Tech roles             :     2
  Baden-Württemberg IT roles  ★       :     0
  Manual portals listed               :    17
─────────────────────────────────────────────────────────────────

── API Fetch Log ──
                     Company   API Type  Jobs Fetched Status
G-P (Globalization Partners) greenhouse            43     OK
                   RemoFirst      lever            13     OK
                  WorkMotion   workable            17     OK

── Sample Raw Data (first 10 rows) ──
                     company             

## 5 · Excel Export

Produces a colour-coded workbook with 9 worksheets:

| # | Sheet | Contents | Header colour |
|---|-------|----------|--------------|
| 1 | **Summary** | Key metrics at a glance | Lavender |
| 2 | **BW_IT_Roles** | IT roles in Baden-Württemberg ★ | Green |
| 3 | **Germany_IT_Roles** | All German IT roles | Blue |
| 4 | **All_IT_Roles** | IT roles — all companies, all locations | Mint |
| 5 | **Germany_All_Roles** | All German roles (any function) | Yellow |
| 6 | **All_Roles_Raw** | Full unfiltered dataset | Grey |
| 7 | **Company_Stats** | Per-company pivot table | Teal |
| 8 | **Manual_Portals** | Companies needing manual search | Orange |
| 9 | **Fetch_Log** | API call results & errors | Pink |

In [37]:
# ── Styling configuration ────────────────────────────────────────────────────
_FILLS = {
    'Summary':           PatternFill('solid', fgColor='D1C4E9'),  # lavender
    'BW_IT_Roles':       PatternFill('solid', fgColor='A5D6A7'),  # green
    'Germany_IT_Roles':  PatternFill('solid', fgColor='90CAF9'),  # blue
    'All_IT_Roles':      PatternFill('solid', fgColor='B2DFDB'),  # mint
    'Germany_All_Roles': PatternFill('solid', fgColor='FFF176'),  # yellow
    'All_Roles_Raw':     PatternFill('solid', fgColor='E0E0E0'),  # grey
    'Company_Stats':     PatternFill('solid', fgColor='80DEEA'),  # teal
    'Manual_Portals':    PatternFill('solid', fgColor='FFCC80'),  # orange
    'Fetch_Log':         PatternFill('solid', fgColor='F8BBD0'),  # pink
}
_HEADER_FONT   = Font(bold=True, color='1A237E', size=11)
_DATA_FONT     = Font(size=10)
_COL_WIDTHS    = {
    'company': 26, 'source_system': 14, 'title': 52,
    'department': 28, 'team': 22, 'location': 28, 'office': 28,
    'remote_type': 14, 'country_code': 14, 'location_blob': 42,
    'region': 36, 'is germany': 14, 'is baden-württemberg': 22,
    'is it / tech role': 18, 'apply_url': 58, 'job_url': 58,
    'updated_at': 24, 'description_snippet': 70, 'collected (utc)': 26,
    'metric': 48, 'value': 14,
    'portal url': 62, 'ats type': 22, 'note': 58,
    'api type': 14, 'jobs found': 12, 'status': 50,
    'company stats': 26,
    'total jobs': 12, 'germany jobs': 14, 'bw jobs': 10,
    'it / tech jobs': 14, 'germany it jobs': 16, 'bw it jobs': 12,
}


def _style_sheet(ws, sheet_name: str) -> None:
    fill = _FILLS.get(sheet_name, PatternFill('solid', fgColor='FFFFFF'))
    for cell in ws[1]:
        cell.font      = _HEADER_FONT
        cell.fill      = fill
        cell.alignment = Alignment(horizontal='center', vertical='center')
    ws.freeze_panes       = 'A2'
    ws.row_dimensions[1].height = 22
    for col in ws.columns:
        hdr   = str(col[0].value or '').lower()
        width = _COL_WIDTHS.get(hdr, 18)
        ws.column_dimensions[get_column_letter(col[0].column)].width = width
        for cell in col[1:]:
            cell.font      = _DATA_FONT
            cell.alignment = Alignment(vertical='top', wrap_text=False)


def _write_sheet(ws, df: pd.DataFrame, sheet_name: str) -> None:
    for r_idx, row in enumerate(dataframe_to_rows(df, index=False, header=True), 1):
        for c_idx, value in enumerate(row, 1):
            ws.cell(row=r_idx, column=c_idx, value=value)
    _style_sheet(ws, sheet_name)


def _build_summary(df_all: pd.DataFrame, df_manual: pd.DataFrame) -> pd.DataFrame:
    g  = df_all['Is Germany']
    bw = df_all['Is Baden-Württemberg']
    it = df_all['Is IT / Tech Role']
    rows = [
        ('Run timestamp (UTC)', datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')),
        ('Companies fetched via API', df_all['company'].nunique()),
        ('Manual portals listed', len(df_manual)),
        ('─── Raw Collection ───', ''),
        ('Total jobs collected (all companies, all locations)', len(df_all)),
        ('Unique job titles', df_all['title'].nunique()),
        ('─── IT / Tech Filter (all locations) ───', ''),
        ('IT / Tech roles — all companies, all locations', int(it.sum())),
        ('─── Germany Filter ───', ''),
        ('Germany jobs — all roles', int(g.sum())),
        ('Germany — Baden-Württemberg (all roles)',   int(bw.sum())),
        ('Germany — Remote / Unspecified (all roles)',
             int((df_all['Region'] == 'Germany — Remote / Unspecified').sum())),
        ('Germany — Other State (all roles)',
             int((df_all['Region'] == 'Germany — Other State').sum())),
        ('─── IT / Tech Filter (Germany) ───', ''),
        ('IT / Tech roles — Germany (all states)',      int((g & it).sum())),
        ('IT / Tech roles — Baden-Württemberg ★', int((bw & it).sum())),
        ('IT / Tech roles — Germany Remote / Unspecified',
             int(((df_all['Region'] == 'Germany — Remote / Unspecified') & it).sum())),
        ('IT / Tech roles — Germany Other State',
             int(((df_all['Region'] == 'Germany — Other State') & it).sum())),
    ]
    return pd.DataFrame(rows, columns=['Metric', 'Value'])


def _build_company_stats(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for co in sorted(df['company'].unique()):
        s = df[df['company'] == co]
        g  = s['Is Germany']
        bw = s['Is Baden-Württemberg']
        it = s['Is IT / Tech Role']
        rows.append({
            'Company':        co,
            'Total Jobs':     len(s),
            'IT / Tech Jobs': int(it.sum()),
            'Germany Jobs':   int(g.sum()),
            'Germany IT Jobs': int((g & it).sum()),
            'BW Jobs':        int(bw.sum()),
            'BW IT Jobs':      int((bw & it).sum()),
        })
    return pd.DataFrame(rows)


def _display_cols(df: pd.DataFrame) -> List[str]:
    preferred = [
        'company', 'title', 'department', 'team',
        'location', 'office', 'remote_type', 'Region',
        'apply_url', 'updated_at', 'source_system',
    ]
    return [c for c in preferred if c in df.columns]


def export_excel(
    df_all: pd.DataFrame,
    df_manual: pd.DataFrame,
    df_log: pd.DataFrame,
) -> None:
    wb = Workbook()
    wb.remove(wb.active)  # remove default blank sheet

    g  = df_all['Is Germany']
    bw = df_all['Is Baden-Württemberg']
    it = df_all['Is IT / Tech Role']

    df_bw_it    = df_all[bw & it].copy()
    df_de_it    = df_all[g  & it].copy()
    df_all_it   = df_all[it].copy()        # all IT roles regardless of country
    df_de_all   = df_all[g].copy()

    sheets = [
        ('Summary',           _build_summary(df_all, df_manual)),
        ('BW_IT_Roles',       df_bw_it[_display_cols(df_bw_it)]   if not df_bw_it.empty  else df_bw_it),
        ('Germany_IT_Roles',  df_de_it[_display_cols(df_de_it)]   if not df_de_it.empty  else df_de_it),
        ('All_IT_Roles',      df_all_it[_display_cols(df_all_it)] if not df_all_it.empty else df_all_it),
        ('Germany_All_Roles', df_de_all[_display_cols(df_de_all)] if not df_de_all.empty else df_de_all),
        ('All_Roles_Raw',     df_all),
        ('Company_Stats',     _build_company_stats(df_all)),
        ('Manual_Portals',    df_manual),
        ('Fetch_Log',         df_log),
    ]

    for sheet_name, df_sheet in sheets:
        ws = wb.create_sheet(sheet_name)
        _write_sheet(ws, df_sheet, sheet_name)

    wb.save(OUTPUT_XLSX)
    print(f'\n✅  Workbook saved: {OUTPUT_XLSX}')
    print(f'   Sheets: {", ".join(s.title for s in wb.worksheets)}')


print('✓ Export functions defined.')

✓ Export functions defined.


In [38]:
export_excel(df_all, df_manual, df_log)


✅  Workbook saved: global_paymaster_IT_jobs_Germany_20260513_1226.xlsx
   Sheets: Summary, BW_IT_Roles, Germany_IT_Roles, All_IT_Roles, Germany_All_Roles, All_Roles_Raw, Company_Stats, Manual_Portals, Fetch_Log


## 6 · Statistics & Summary

All figures below reflect what was retrievable via public APIs at run time.  
Companies whose APIs returned errors are listed in **Manual_Portals** in the Excel workbook.

In [39]:
from IPython.display import display, HTML

print('\n' + '=' * 65)
print('  OVERALL COLLECTION SUMMARY')
print('=' * 65)

df_summary_display = _build_summary(df_all, df_manual)

# Render as a plain HTML table — avoids pandas 3.x Styler API changes
display(HTML(
    df_summary_display.to_html(index=False, border=0, classes='summary-tbl')
))


  OVERALL COLLECTION SUMMARY


Metric,Value
Run timestamp (UTC),2026-05-13 10:26:45
Companies fetched via API,3
Manual portals listed,17
─── Raw Collection ───,
"Total jobs collected (all companies, all locations)",61
Unique job titles,50
─── IT / Tech Filter (all locations) ───,
"IT / Tech roles — all companies, all locations",21
─── Germany Filter ───,
Germany jobs — all roles,5


In [40]:
print('\n' + '=' * 65)
print('  PER-COMPANY BREAKDOWN')
print('=' * 65)

cs = _build_company_stats(df_all)
if not cs.empty:
    display(HTML(cs.to_html(index=False, border=0, classes='cs-tbl')))
    print(f'\n  Total companies with API data : {len(cs)}')
    print(f'  Companies with any IT roles   : {(cs["IT / Tech Jobs"] > 0).sum()}')
    print(f'  Companies with Germany IT roles: {(cs["Germany IT Jobs"] > 0).sum()}')
    print(f'  Companies with BW IT roles     : {(cs["BW IT Jobs"] > 0).sum()}')
else:
    print('No API data collected.')


  PER-COMPANY BREAKDOWN


Company,Total Jobs,IT / Tech Jobs,Germany Jobs,Germany IT Jobs,BW Jobs,BW IT Jobs
G-P (Globalization Partners),43,18,0,0,0,0
RemoFirst,13,1,0,0,0,0
WorkMotion,5,2,5,2,0,0



  Total companies with API data : 3
  Companies with any IT roles   : 3
  Companies with Germany IT roles: 1
  Companies with BW IT roles     : 0


In [41]:
df_all_it_display = df_all[df_all['Is IT / Tech Role']].copy()

print('\n' + '=' * 65)
print(f'  ALL IT / TECH ROLES — ALL COMPANIES, ALL LOCATIONS  ({len(df_all_it_display)} found)')
print('=' * 65)

if df_all_it_display.empty:
    print('\n  ⚠  No IT / tech roles found via public APIs.')
    print('     All API sources may have failed — check Fetch_Log in the Excel workbook.')
else:
    show_cols = [c for c in ['company', 'title', 'department', 'location', 'Region', 'remote_type', 'apply_url']
                 if c in df_all_it_display.columns]
    display(HTML(
        df_all_it_display[show_cols]
            .sort_values(['company', 'title'])
            .reset_index(drop=True)
            .to_html(index=False, border=0)
    ))


  ALL IT / TECH ROLES — ALL COMPANIES, ALL LOCATIONS  (21 found)


company,title,department,location,Region,remote_type,apply_url
G-P (Globalization Partners),"AI, Intern",Technology,United States (Remote-First),Non-Germany,,https://job-boards.greenhouse.io/globalizationpartners/jobs/7720641003
G-P (Globalization Partners),Global IT Cloud Systems Manager,IT,United States (Remote-First),Non-Germany,,https://job-boards.greenhouse.io/globalizationpartners/jobs/7662992003
G-P (Globalization Partners),Information Security Engineer,IT,United Kingdom (Remote-First),Non-Germany,,https://job-boards.greenhouse.io/globalizationpartners/jobs/7681932003
G-P (Globalization Partners),Information Security Engineer,IT,Ireland (Remote-First),Non-Germany,,https://job-boards.greenhouse.io/globalizationpartners/jobs/7681929003
G-P (Globalization Partners),Information Security Engineer,IT,United Kingdom - Northern Ireland (Remote-First),Non-Germany,,https://job-boards.greenhouse.io/globalizationpartners/jobs/7681933003
G-P (Globalization Partners),Senior Data Scientist & Engineer,Technology,India (Remote-First),Non-Germany,,https://job-boards.greenhouse.io/globalizationpartners/jobs/7704107003
G-P (Globalization Partners),Senior Software Engineer,Technology,Ireland (Remote-First),Non-Germany,,https://job-boards.greenhouse.io/globalizationpartners/jobs/7698181003
G-P (Globalization Partners),Senior Software Engineer,Technology,United Kingdom - Northern Ireland (Remote-First),Non-Germany,,https://job-boards.greenhouse.io/globalizationpartners/jobs/7698185003
G-P (Globalization Partners),Senior Software Engineer,Technology,India (Remote-First),Non-Germany,,https://job-boards.greenhouse.io/globalizationpartners/jobs/7698180003
G-P (Globalization Partners),Senior Software Engineer (Data Platform),Technology,India (Remote-First),Non-Germany,,https://job-boards.greenhouse.io/globalizationpartners/jobs/7698156003


In [42]:
df_bw_it = df_all[df_all['Is Baden-Württemberg'] & df_all['Is IT / Tech Role']].copy()

print('\n' + '=' * 65)
print(f'  IT ROLES IN BADEN-WÜRTTEMBERG  ★  ({len(df_bw_it)} found)')
print('=' * 65)

if df_bw_it.empty:
    print('\n  ⚠  No Baden-Württemberg IT roles found via public APIs.')
    print('     Search the Manual_Portals sheet for companies to check manually.')
else:
    show_cols = [c for c in ['company', 'title', 'department', 'location', 'remote_type', 'apply_url']
                 if c in df_bw_it.columns]
    display(HTML(
        df_bw_it[show_cols]
            .sort_values(['company', 'title'])
            .reset_index(drop=True)
            .to_html(index=False, border=0)
    ))


  IT ROLES IN BADEN-WÜRTTEMBERG  ★  (0 found)

  ⚠  No Baden-Württemberg IT roles found via public APIs.
     Search the Manual_Portals sheet for companies to check manually.


In [43]:
df_de_nob = df_all[
    df_all['Is Germany'] &
    df_all['Is IT / Tech Role'] &
    ~df_all['Is Baden-Württemberg']
].copy()

print('\n' + '=' * 65)
print(f'  IT ROLES IN GERMANY — OTHER STATES / REMOTE  ({len(df_de_nob)} found)')
print('=' * 65)

if df_de_nob.empty:
    print('\n  No IT roles in Germany (non-BW) found via public APIs.')
else:
    show_cols = [c for c in ['company', 'title', 'department', 'location', 'Region', 'apply_url']
                 if c in df_de_nob.columns]
    display(HTML(
        df_de_nob[show_cols]
            .sort_values(['Region', 'company', 'title'])
            .reset_index(drop=True)
            .to_html(index=False, border=0)
    ))

# ── Manual portals reminder ──────────────────────────────────────────────────
print('\n' + '=' * 65)
print(f'  MANUAL PORTALS TO CHECK  ({len(df_manual)} companies)')
print('=' * 65)
print('  These companies do not expose a public job API.')
print('  Visit each URL and search for IT roles in Germany / Baden-Württemberg.\n')
display(HTML(df_manual.to_html(index=False, border=0)))

print('\n' + '=' * 65)
print(f'  ✅  DONE.  Workbook: {OUTPUT_XLSX}')
print('=' * 65)


  IT ROLES IN GERMANY — OTHER STATES / REMOTE  (2 found)


company,title,department,location,Region,apply_url
WorkMotion,GTM Engineer (Revenue Systems & Automation),WorkMotion,Germany (Remote),Germany — Remote / Unspecified,https://apply.workable.com/j/362071F409/apply
WorkMotion,Principal Engineer | Software Architect,,Germany (Remote),Germany — Remote / Unspecified,https://apply.workable.com/j/90894BA5DC/apply



  MANUAL PORTALS TO CHECK  (17 companies)
  These companies do not expose a public job API.
  Visit each URL and search for IT roles in Germany / Baden-Württemberg.



Company,Portal URL,ATS Type,Note
Deel,https://www.deel.com/careers/,Manual / Workday / Proprietary,No verified public API. Search portal manually for IT roles in Germany.
Remote,https://remote.com/careers,Manual / Workday / Proprietary,No verified public API. Search portal manually for IT roles in Germany.
Rippling,https://www.rippling.com/careers,Manual / Workday / Proprietary,No verified public API. Search portal manually for IT roles in Germany.
Oyster HR,https://www.oysterhr.com/careers,Manual / Workday / Proprietary,No verified public API. Search portal manually for IT roles in Germany.
Velocity Global,https://velocityglobal.com/company/careers/,Manual / Workday / Proprietary,No verified public API. Search portal manually for IT roles in Germany.
Omnipresent,https://omnipresent.com/careers/,Manual / Workday / Proprietary,No verified public API. Search portal manually for IT roles in Germany.
Atlas HXM,https://www.atlashxm.com/careers,Manual / Workday / Proprietary,No verified public API. Search portal manually for IT roles in Germany.
Papaya Global,https://www.papayaglobal.com/careers/,Manual / Workday / Proprietary,No verified public API. Search portal manually for IT roles in Germany.
Multiplier,https://www.usemultiplier.com/careers,Manual / Workday / Proprietary,No verified public API. Search portal manually for IT roles in Germany.
Safeguard Global,https://safeguardglobal.wd3.myworkdayjobs.com/External_Careers/,Manual / Workday / Proprietary,No verified public API. Search portal manually for IT roles in Germany.



  ✅  DONE.  Workbook: global_paymaster_IT_jobs_Germany_20260513_1226.xlsx
